# Exercício 8 — HTML, `requests` e BeautifulSoup

Este exercício repete o fluxo da aula (requisitar → checar status → inspecionar → extrair → conferir → salvar), agora numa página real escolhida por você, dentro do que é permitido raspar.

Páginas sugeridas, feitas justamente para prática de scraping (com permissão explícita para isso):

- **http://books.toscrape.com**: catálogo fake de livros, com título, preço e link de cada um.
- **http://quotes.toscrape.com**: lista de citações, com texto, autor e tags.

Também pode usar outra página, desde que você tenha certeza de que pode raspar (confira `robots.txt` e os termos de uso, seção 15 da aula). Na dúvida, use uma das duas sugeridas.

Antes de TUDO, copie a pasta `exercicios/` para dentro da sua pasta de entregas (`extracao-dados-trabalhos-SeuNome`, com o SEU nome), dentro de uma pasta `08-html-requests-beautifulsoup` em `projetos/`.

## Preparação do ambiente

Como este notebook vai morar numa pasta própria (dentro da sua pasta de entregas), use o `.venv` da **raiz do seu repositório de trabalhos**. Não precisa criar um `.venv` novo dentro de `exercicios/`.

Na raiz do repositório de trabalhos:

```cmd
uv pip install -r projetos/08-html-requests-beautifulsoup/exercicios/requirements.txt
```

(Ajuste o caminho se a pasta da entrega tiver outro nome. O arquivo precisa de `requests` e `beautifulsoup4`.)

Se o `uv` não funcionar, use `pip install -r ...` com o ambiente já ativado.


## Parte 0 — Escolha da página

Escolha uma página real para raspar (uma das sugeridas, ou outra que você tenha certeza que pode raspar) e registre aqui antes de começar:

**Página escolhida (nome e URL):**

> Escreva aqui.

**Por que essa coleta é permitida (o que você conferiu em `robots.txt` ou nos termos de uso):**

> Escreva aqui.

**O que você quer extrair dela (que campos, tipo título, preço, autor, link):**

> Escreva aqui.

## Parte 1 — Inspecionar no navegador

Antes de escrever qualquer código, abra a página escolhida no navegador e inspecione o HTML (botão direito → Inspecionar, ou `Ctrl+Shift+I` / `Cmd+Option+I`, seção 4 da aula). Encontre o elemento que repete para cada item da lista (cada livro, cada citação) e anote a **tag**, a **classe** e, se fizer sentido, o **id**.

**Elemento que repete (ex.: tag `article` com classe `product_pod`, ou tag `div` com classe `quote`):**

> Escreva aqui.

**Campos dentro dele (ex.: título no `h3` → `a`, preço no `p` com classe `price_color`):**

> Escreva aqui.


## Parte 2 — Requisição e checagem do status

Complete a URL abaixo com a página escolhida na Parte 0 e faça a requisição. **Não siga para a Parte 3 se o status não for 200:** revise a URL, a conexão, ou se a página está mesmo disponível.

In [ ]:
import requests  # biblioteca de requisição HTTP

url = ""  # cole aqui a URL escolhida na Parte 0, por exemplo "http://books.toscrape.com/"

resposta = requests.get(url)  # faz a requisição GET
print(f"Status: {resposta.status_code}")

if resposta.status_code == 200:
    resposta.encoding = resposta.apparent_encoding  # corrige a codificação de texto (evita símbolo estranho no lugar de acento)
    html_pagina = resposta.text
    print(f"HTML recebido: {len(html_pagina)} caracteres")
else:
    html_pagina = None
    print("A página não respondeu como esperado. Revise a URL antes de continuar.")

## Parte 3 — Parsear com BeautifulSoup

Transforme o HTML recebido numa árvore navegável.

In [ ]:
from bs4 import BeautifulSoup

sopa = BeautifulSoup(html_pagina, "html.parser") if html_pagina else None

if sopa:
    print(sopa.title.get_text())  # confirma que o parse funcionou, mostrando o título da página

## Parte 4 — Extraindo os campos

Use `.find_all()` no elemento que repete (anotado na Parte 1) para percorrer todos os itens da página, e `.find()` em cada campo para extrair o que interessa. Siga o modelo das Seções 12 e 14 da aula: uma lista vazia, um `for` que percorre os itens, um dicionário por item.

Complete os `...` abaixo com as suas tags/classes e nomes de campo. Use `.get_text(strip=True)` para texto e `elemento["href"]` (ou outro atributo) para links.


In [ ]:
itens_extraidos = []  # lista vazia que vai guardar um dicionário por item

if sopa:
    for item in sopa.find_all("...", {"class": "..."}):  # troque pelos tag/classe do elemento que repete (Parte 1)
        # troque os "..." abaixo pelas buscas de cada campo, e os nomes de chave pelos campos da Parte 0
        campo_1 = item.find("...", {"class": "..."}).get_text(strip=True)
        campo_2 = item.find("...", {"class": "..."}).get_text(strip=True)
        # se algum campo for um link, use item.find("a")["href"] (ou a tag certa) em vez de .get_text()
        # se a tag não tiver classe, pode ser só item.find("h3") ou item.find("h3").find("a")

        itens_extraidos.append({
            "campo_1": campo_1,
            "campo_2": campo_2,
        })

print(f"Total extraído: {len(itens_extraidos)}")


## Parte 5 — Conferir

Antes de salvar, confira o resultado: quantos itens vieram, e como estão os primeiros registros. Se `len(itens_extraidos)` for `0`, revise a tag/classe da Parte 1 e da Parte 4 antes de seguir (seção 16 da aula, "Quando der errado").


In [ ]:
print(f"Total de itens: {len(itens_extraidos)}")
itens_extraidos[:5]  # espia os cinco primeiros, pra conferir que os campos vieram certos

## Parte 6 — Salvar em CSV

Salve o resultado em `dados/coleta.csv`, com colunas nomeadas (os mesmos nomes de campo da Parte 4).

In [ ]:
import csv
from pathlib import Path

Path("dados").mkdir(exist_ok=True)  # garante que a pasta dados/ existe

with open("dados/coleta.csv", "w", encoding="utf-8", newline="") as arquivo:
    colunas = list(itens_extraidos[0].keys()) if itens_extraidos else []  # usa as chaves do primeiro dicionário como colunas
    escritor = csv.DictWriter(arquivo, fieldnames=colunas)
    escritor.writeheader()
    escritor.writerows(itens_extraidos)

print("Salvo em dados/coleta.csv")

## Parte 7 — Registro da coleta

Crie `projetos/08-html-requests-beautifulsoup/README.md` (na sua pasta de entregas, não aqui) e responda:

**Fonte (nome da página e URL):**

> Escreva aqui.

**Tag/classe/id usados no `.find()` / `.find_all()` para extrair os dados:**

> Escreva aqui.

**Data e hora da coleta:**

> Escreva aqui.

**Falhas encontradas durante a coleta (status diferente de 200, busca que não bateu de primeira, item sem algum campo, etc.) e como foram resolvidas:**

> Escreva aqui.

**Declaração de uso de IA:** ferramenta usada, em que trecho ou decisão desta entrega, e o que você conferiu ou alterou depois do resultado gerado (mesmo que a resposta seja "não usei IA nesta entrega", registre isso).


## Parte 8 — Conferência final

Antes de considerar a entrega concluída, confira:

- [ ] A Parte 0 registra a página escolhida e por que a coleta é permitida.
- [ ] A Parte 1 registra a tag/classe/id anotados a partir da inspeção no navegador.
- [ ] A Parte 2 verificou o `status_code` da requisição e só seguiu adiante porque o status era `200`.
- [ ] A Parte 4 rodou sem erro e extraiu mais de um item.
- [ ] A Parte 5 mostra o total de itens extraídos e os cinco primeiros registros.
- [ ] `dados/coleta.csv` foi gerado, com colunas nomeadas, e a pasta `dados/` está no `.gitignore`.
- [ ] `projetos/08-html-requests-beautifulsoup/README.md` responde às quatro perguntas da Parte 7, incluindo a declaração de uso de IA.
- [ ] O notebook roda do início ao fim sem erro usando Kernel → Restart e Run All (ou o equivalente no VS Code), não só célula por célula fora de ordem.
- [ ] Esse notebook está copiado dentro de `projetos/08-html-requests-beautifulsoup/` na sua pasta de entregas.
- [ ] Você já fez `git add`, `git commit` e `git push` dessa entrega.
